# 04 — Modeling RH: Attrition (Probabilidade de Rotatividade)

**Objetivo:** treinar um modelo supervisionado para estimar a probabilidade de `Rotatividade`.

Entradas:
- `data/interim/rh_joined.csv`

Saídas:
- `artifacts/model_attrition.pkl`
- métricas e interpretação (top fatores)


### 4.1 Setup

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

### 4.2 Load (interim) e target binário

In [ ]:
DATA_INTERIM = Path("../data/interim")
ARTIFACTS = Path("../artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_INTERIM / "rh_joined.csv")

TARGET_COL = "Rotatividade"
df["rotatividade_bin"] = (df[TARGET_COL].astype(str).str.lower() == "sim").astype(int)

df["rotatividade_bin"].value_counts()

### 4.3 Seleção de colunas (anti-leakage)

Remova colunas que só existiriam após o desligamento.


In [ ]:
KEY = "IDDoEmpregado"

drop_cols = [TARGET_COL, "rotatividade_bin"]

X = df.drop(columns=drop_cols, errors="ignore")
y = df["rotatividade_bin"]

# remover ID (tende a ser ruído)
if KEY in X.columns:
    X = X.drop(columns=[KEY])

X.shape, y.shape

### 4.4 Split estratificado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

y_train.mean().round(4), y_test.mean().round(4)

### 4.5 Preprocessador dentro do pipeline (evita vazamento)

- numéricas: mediana
- categóricas: moda + one-hot


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"))
        ]), cat_cols)
    ],
    remainder="drop"
)

model = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)

clf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", model)
])

clf

### 4.6 Treino

In [ ]:
clf.fit(X_train, y_train)

### 4.7 Avaliação (AUC + métricas)

In [ ]:
proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba)
print("AUC:", round(float(auc), 4))
print(classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d")
plt.title("Confusion Matrix")
plt.xlabel("Pred")
plt.ylabel("True")
plt.show()

RocCurveDisplay.from_predictions(y_test, proba)
plt.show()

### 4.8 Interpretabilidade — Top fatores

Para Logística:
- coeficiente > 0 aumenta risco
- coeficiente < 0 reduz risco


In [ ]:
# Recupera nomes após OneHot
ohe = clf.named_steps["prep"].named_transformers_["cat"].named_steps["ohe"]
cat_feature_names = ohe.get_feature_names_out(cat_cols)
feature_names = np.r_[num_cols, cat_feature_names]

coefs = clf.named_steps["model"].coef_[0]

imp = (pd.DataFrame({"feature": feature_names, "coef": coefs})
         .assign(abs_coef=lambda d: d["coef"].abs())
         .sort_values("abs_coef", ascending=False))

imp.head(25)

### 4.9 Persistência do modelo

In [ ]:
joblib.dump(clf, ARTIFACTS / "model_attrition.pkl")
print("OK ->", ARTIFACTS / "model_attrition.pkl")

### 4.10 Conclusão

Este modelo é o baseline explicável. Próximos upgrades possíveis:
- calibrar threshold (business-driven)
- comparar com RandomForest / XGBoost
- análise de erro por segmento
